In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

spark.stop()

spark = SparkSession.builder \
    .appName("FRAUD-X42 NexBuy") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} initialise en mode local")

Spark 4.1.2 initialise en mode local


In [ ]:
# Chargement via pandas puis conversion Spark
import pandas as pd
import os

path = os.path.abspath("nexbuy_transactions.csv")
pdf = pd.read_csv(path)

print(f"Lignes : {len(pdf)}")
print(f"Colonnes : {len(pdf.columns)}")
print(pdf.dtypes)
print(pdf.head(5))

Lignes : 4970
Colonnes : 16
transaction_id          str
timestamp               str
user_id                 str
session_id              str
amount_eur          float64
product_category        str
payment_method          str
country_code            str
device_type             str
ip_address              str
nb_items              int64
is_new_account         bool
account_age_days      int64
delivery_country        str
billing_country         str
is_fraud               bool
dtype: object
  transaction_id            timestamp   user_id  session_id  amount_eur  \
0      TXN-00001  2026-03-09 22:36:31  USR-0021  SESS-38676       23.56   
1      TXN-00002  2026-03-06 23:25:24  USR-0309  SESS-74244       26.57   
2      TXN-00003  2026-04-18 22:50:00  USR-0105  SESS-46367       49.08   
3      TXN-00004  2026-04-02 17:14:29  USR-0012  SESS-75789       62.13   
4      TXN-00005  2026-03-21 22:57:43  USR-0596  SESS-10697      205.49   

  product_category  payment_method country_code device_type

In [ ]:
# Conversion en Spark DataFrame
df = spark.createDataFrame(pdf)

print("SCHEMA")
df.printSchema()

print(f"Lignes : {df.count()}")
df.show(5, truncate=False)

SCHEMA
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- nb_items: long (nullable = true)
 |-- is_new_account: boolean (nullable = true)
 |-- account_age_days: long (nullable = true)
 |-- delivery_country: string (nullable = true)
 |-- billing_country: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)



Lignes : 4970
+--------------+-------------------+--------+----------+----------+----------------+--------------+------------+-----------+--------------+--------+--------------+----------------+----------------+---------------+--------+
|transaction_id|timestamp          |user_id |session_id|amount_eur|product_category|payment_method|country_code|device_type|ip_address    |nb_items|is_new_account|account_age_days|delivery_country|billing_country|is_fraud|
+--------------+-------------------+--------+----------+----------+----------------+--------------+------------+-----------+--------------+--------+--------------+----------------+----------------+---------------+--------+
|TXN-00001     |2026-03-09 22:36:31|USR-0021|SESS-38676|23.56     |Sport           |paypal        |ES          |desktop    |156.243.221.9 |6       |false         |384             |ES              |ES             |false   |
|TXN-00002     |2026-03-06 23:25:24|USR-0309|SESS-74244|26.57     |Beaute          |carte_banc

Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [10]:
# Exploration des valeurs manquantes et aberrantes
from pyspark.sql.functions import col, count, when, isnan

print("VALEURS MANQUANTES PAR COLONNE")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(truncate=False)

print("STATISTIQUES amount_eur")
df.select("amount_eur").describe().show()

print("MONTANTS ABERRANTS (negatifs ou > 10000)")
df.filter((col("amount_eur") <= 0) | (col("amount_eur") > 10000)).count()

VALEURS MANQUANTES PAR COLONNE
+--------------+---------+-------+----------+----------+----------------+--------------+------------+-----------+----------+--------+--------------+----------------+----------------+---------------+--------+
|transaction_id|timestamp|user_id|session_id|amount_eur|product_category|payment_method|country_code|device_type|ip_address|nb_items|is_new_account|account_age_days|delivery_country|billing_country|is_fraud|
+--------------+---------+-------+----------+----------+----------------+--------------+------------+-----------+----------+--------+--------------+----------------+----------------+---------------+--------+
|0             |0        |0      |0         |0         |0               |0             |0           |0          |0         |0       |0             |0               |0               |0              |0       |
+--------------+---------+-------+----------+----------+----------------+--------------+------------+-----------+----------+--------+----

8

In [11]:
# Pretraitement
from pyspark.sql.functions import to_timestamp, hour, dayofweek

# Filtrer les montants aberrants
df_clean = df.filter((col("amount_eur") > 0) & (col("amount_eur") <= 10000))

# Convertir le timestamp
df_clean = df_clean.withColumn("timestamp", to_timestamp(col("timestamp")))

# Extraire heure et jour
df_clean = df_clean.withColumn("hour", hour(col("timestamp")))
df_clean = df_clean.withColumn("day_of_week", dayofweek(col("timestamp")))

# Mettre en cache pour optimisation
df_clean.cache()

print(f"Lignes apres nettoyage : {df_clean.count()}")
print(f"Lignes supprimees : {df.count() - df_clean.count()}")
df_clean.printSchema()

Lignes apres nettoyage : 4962
Lignes supprimees : 8
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- nb_items: long (nullable = true)
 |-- is_new_account: boolean (nullable = true)
 |-- account_age_days: long (nullable = true)
 |-- delivery_country: string (nullable = true)
 |-- billing_country: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)



In [13]:
# Cellule 7 — Piste A : visualisation + heures suspectes seuil 1.5
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("HEURES SUSPECTES (z-score > 1.5)")
hourly_zscore.filter(col("zscore") > 1.5).show(truncate=False)

# Visualisation
hourly_pd = hourly_zscore.toPandas()

plt.figure(figsize=(14, 5))
colors = ['red' if z > 1.5 else 'steelblue' for z in hourly_pd['zscore']]
plt.bar(hourly_pd['hour'], hourly_pd['count'], color=colors)
plt.axhline(y=mean_val, color='orange', linestyle='--', label='Moyenne')
plt.title('Piste A — Volume de transactions par heure')
plt.xlabel('Heure')
plt.ylabel('Nombre de transactions')
plt.legend(['Moyenne', 'Normal', 'Suspect (z>1.5)'])
plt.xticks(range(24))
plt.tight_layout()
plt.savefig('piste_a_horaire.png', dpi=150)
plt.show()
print("Graphique sauvegarde : piste_a_horaire.png")

HEURES SUSPECTES (z-score > 1.5)
+----+-----+------------------+
|hour|count|zscore            |
+----+-----+------------------+
|2   |233  |1.8365958986809292|
|11  |233  |1.8365958986809292|
+----+-----+------------------+

Graphique sauvegarde : piste_a_horaire.png


In [ ]:
# Piste B : Burst client
from pyspark.sql.functions import unix_timestamp, lag, sum as spark_sum
from pyspark.sql.window import Window
import builtins

# Window par user ordonne par timestamp
w_user = Window.partitionBy("user_id").orderBy("timestamp")

# Transactions en moins de 2h
df_burst = df_clean.withColumn(
    "prev_ts",
    lag(unix_timestamp("timestamp")).over(w_user)
)

df_burst = df_burst.withColumn(
    "diff_seconds",
    unix_timestamp("timestamp") - col("prev_ts")
)

bursts = df_burst.filter(
    col("diff_seconds").isNotNull() &
    (col("diff_seconds") < 7200)
).groupBy("user_id").count().filter(col("count") > 5)

print("CLIENTS BURST > 5 TRANSACTIONS EN MOINS DE 2H")
bursts.orderBy(col("count").desc()).show(20, truncate=False)

# Cumul > 1500 EUR en 24h via pandas
print("CLIENTS CUMUL > 1500 EUR EN 24H")
burst_pd = df_clean.select(
    "user_id", "transaction_id", "timestamp", "amount_eur"
).toPandas()

burst_pd["timestamp"] = pd.to_datetime(burst_pd["timestamp"])
burst_pd = burst_pd.sort_values(["user_id", "timestamp"])

results = []
for user, group in burst_pd.groupby("user_id"):
    for i, row in group.iterrows():
        window = group[
            (group["timestamp"] >= row["timestamp"] - pd.Timedelta(hours=24)) &
            (group["timestamp"] <= row["timestamp"])
        ]
        cumul = window["amount_eur"].sum()
        if cumul > 1500:
            results.append({
                "user_id": user,
                "transaction_id": row["transaction_id"],
                "cumul_24h": builtins.round(float(cumul), 2),
                "timestamp": row["timestamp"]
            })

cumul_df = pd.DataFrame(results).drop_duplicates("user_id").sort_values(
    "cumul_24h", ascending=False
)
print(cumul_df.head(20).to_string(index=False))
print(f"\nTotal clients suspects cumul : {len(cumul_df)}")

CLIENTS BURST > 5 TRANSACTIONS EN MOINS DE 2H
+--------+-----+
|user_id |count|
+--------+-----+
|USR-0175|11   |
|USR-0194|10   |
|USR-0063|9    |
|USR-0078|8    |
|USR-0415|8    |
|USR-0301|7    |
|USR-0523|7    |
|USR-0432|6    |
+--------+-----+

CLIENTS CUMUL > 1500 EUR EN 24H
 user_id transaction_id  cumul_24h           timestamp
USR-0389      TXN-03976    2359.56 2026-04-10 03:06:25
USR-0112      TXN-00917    2176.74 2026-03-03 19:47:20
USR-0156      TXN-04728    2127.02 2026-04-08 19:05:48
USR-0267      TXN-03868    2108.00 2026-03-04 23:07:46
USR-0318      TXN-01855    2023.69 2026-03-29 00:41:30
USR-0175      TXN-01164    1953.42 2026-03-14 11:49:45
USR-0432      TXN-00420    1880.03 2026-04-15 03:16:01
USR-0415      TXN-02758    1853.09 2026-04-12 08:11:59
USR-0047      TXN-03680    1807.82 2026-03-31 10:28:33
USR-0373      TXN-00024    1793.11 2026-03-14 00:28:24
USR-0288      TXN-03969    1769.06 2026-04-29 14:28:52
USR-0334      TXN-03321    1758.60 2026-05-12 02:39:43
US

In [18]:
# visualisation Piste B
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 — Burst transactions
bursts_pd = bursts.orderBy(col("count").desc()).toPandas()
ax1.barh(bursts_pd["user_id"], bursts_pd["count"], color="tomato")
ax1.axvline(x=5, color="black", linestyle="--", label="Seuil 5")
ax1.set_title("Clients — Burst transactions < 2h")
ax1.set_xlabel("Nombre de transactions")
ax1.legend()

# Graphique 2 — Cumul 24h
ax2.barh(cumul_df.head(10)["user_id"], cumul_df.head(10)["cumul_24h"], color="orange")
ax2.axvline(x=1500, color="black", linestyle="--", label="Seuil 1500 EUR")
ax2.set_title("Clients — Cumul > 1500 EUR en 24h")
ax2.set_xlabel("Montant cumule (EUR)")
ax2.legend()

plt.tight_layout()
plt.savefig("piste_b_burst.png", dpi=150)
plt.show()
print("Graphique sauvegarde : piste_b_burst.png")

Graphique sauvegarde : piste_b_burst.png


In [19]:
# Piste C : Geographie suspecte
pays_risque = ["RO", "UA", "MD", "BY", "XK"]

# Filtre : billing != delivery + paiement suspect
geo_suspects = df_clean.filter(
    (col("billing_country") != col("delivery_country")) &
    (col("payment_method").isin(["carte_prepayee", "crypto"]))
)

print(f"Transactions suspectes (geo + paiement) : {geo_suspects.count()}")

# Repartition par pays de livraison
print("REPARTITION PAR PAYS DE LIVRAISON")
geo_suspects.groupBy("delivery_country") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

# Croiser avec pays a risque
print("TRANSACTIONS VERS PAYS A RISQUE")
geo_risque = geo_suspects.filter(col("delivery_country").isin(pays_risque))
print(f"Total : {geo_risque.count()}")
geo_risque.groupBy("delivery_country", "payment_method") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

Transactions suspectes (geo + paiement) : 170
REPARTITION PAR PAYS DE LIVRAISON
+----------------+-----+
|delivery_country|count|
+----------------+-----+
|XK              |40   |
|UA              |38   |
|RO              |37   |
|BY              |30   |
|MD              |25   |
+----------------+-----+

TRANSACTIONS VERS PAYS A RISQUE
Total : 170
+----------------+--------------+-----+
|delivery_country|payment_method|count|
+----------------+--------------+-----+
|XK              |carte_prepayee|30   |
|RO              |carte_prepayee|29   |
|UA              |carte_prepayee|26   |
|BY              |carte_prepayee|20   |
|MD              |carte_prepayee|17   |
|UA              |crypto        |12   |
|BY              |crypto        |10   |
|XK              |crypto        |10   |
|MD              |crypto        |8    |
|RO              |crypto        |8    |
+----------------+--------------+-----+



In [20]:
# Cellule 11 — Piste C : visualisation
geo_pd = geo_suspects.groupBy("delivery_country", "payment_method") \
    .count().toPandas()

geo_pivot = geo_pd.pivot(index="delivery_country", columns="payment_method", values="count").fillna(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 — Par pays
geo_total = geo_pd.groupby("delivery_country")["count"].sum().sort_values(ascending=False)
ax1.bar(geo_total.index, geo_total.values, color="crimson")
ax1.set_title("Piste C — Transactions suspectes par pays")
ax1.set_xlabel("Pays de livraison")
ax1.set_ylabel("Nombre de transactions")

# Graphique 2 — Par mode de paiement
geo_pivot.plot(kind="bar", ax=ax2, color=["steelblue", "orange"])
ax2.set_title("Piste C — Mode de paiement par pays")
ax2.set_xlabel("Pays de livraison")
ax2.set_ylabel("Nombre de transactions")
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig("piste_c_geo.png", dpi=150)
plt.show()
print("Graphique sauvegarde : piste_c_geo.png")

Graphique sauvegarde : piste_c_geo.png


In [21]:
# Cellule 12 — Piste D : Nouveaux comptes vs anciens
import numpy as np

# Separer nouveaux (<7 jours) et anciens comptes
nouveaux = df_clean.filter(col("account_age_days") < 7)
anciens = df_clean.filter(col("account_age_days") >= 7)

print(f"Nouveaux comptes (<7j) : {nouveaux.count()}")
print(f"Anciens comptes (>=7j) : {anciens.count()}")

# Statistiques comparatives
print("\nSTATISTIQUES NOUVEAUX COMPTES")
nouveaux.select("amount_eur").describe().show()

print("STATISTIQUES ANCIENS COMPTES")
anciens.select("amount_eur").describe().show()

# Montants eleves chez nouveaux comptes
print("NOUVEAUX COMPTES AVEC MONTANT > 500 EUR")
nouveaux.filter(col("amount_eur") > 500) \
    .select("transaction_id", "user_id", "amount_eur", "payment_method", "delivery_country") \
    .orderBy(col("amount_eur").desc()) \
    .show(20, truncate=False)

Nouveaux comptes (<7j) : 142
Anciens comptes (>=7j) : 4820

STATISTIQUES NOUVEAUX COMPTES
+-------+-----------------+
|summary|       amount_eur|
+-------+-----------------+
|  count|              142|
|   mean|905.2585211267605|
| stddev|664.2024181565673|
|    min|           152.31|
|    max|          2498.73|
+-------+-----------------+

STATISTIQUES ANCIENS COMPTES
+-------+-----------------+
|summary|       amount_eur|
+-------+-----------------+
|  count|             4820|
|   mean|66.77087344398345|
| stddev|80.66286515869388|
|    min|              5.0|
|    max|           1758.6|
+-------+-----------------+

NOUVEAUX COMPTES AVEC MONTANT > 500 EUR
+--------------+--------+----------+--------------+----------------+
|transaction_id|user_id |amount_eur|payment_method|delivery_country|
+--------------+--------+----------+--------------+----------------+
|TXN-01271     |USR-0112|2498.73   |crypto        |XK              |
|TXN-02812     |USR-0267|2498.62   |carte_prepayee|BY      

In [22]:
# Piste D : visualisation
nouveaux_pd = nouveaux.select("amount_eur").toPandas()
anciens_pd = anciens.select("amount_eur").toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot comparatif
ax1.boxplot(
    [nouveaux_pd["amount_eur"], anciens_pd["amount_eur"]],
    labels=["Nouveaux (<7j)", "Anciens (>=7j)"],
    patch_artist=True,
    boxprops=dict(facecolor="tomato"),
)
ax1.set_title("Piste D — Distribution montants par age du compte")
ax1.set_ylabel("Montant (EUR)")

# Histogramme nouveaux comptes
ax2.hist(nouveaux_pd["amount_eur"], bins=20, color="tomato", edgecolor="black")
ax2.axvline(x=500, color="black", linestyle="--", label="Seuil 500 EUR")
ax2.set_title("Piste D — Distribution montants nouveaux comptes")
ax2.set_xlabel("Montant (EUR)")
ax2.set_ylabel("Frequence")
ax2.legend()

plt.tight_layout()
plt.savefig("piste_d_comptes.png", dpi=150)
plt.show()
print("Graphique sauvegarde : piste_d_comptes.png")

Graphique sauvegarde : piste_d_comptes.png


In [24]:
# Cellule 14 — Piste E : Score composite 0-100
import builtins

# Collecter les IDs suspects de chaque piste
burst_users = set(bursts.toPandas()["user_id"].tolist())
cumul_users = set(cumul_df["user_id"].tolist())
geo_txns = set(geo_risque.toPandas()["transaction_id"].tolist())
new_account_txns = set(
    nouveaux.filter(col("amount_eur") > 500)
    .toPandas()["transaction_id"].tolist()
)
heure_suspecte = [2, 11]

# Construire le score sur chaque transaction
df_score = df_clean.toPandas()

def compute_score(row):
    score = 0

    # Piste A — heure suspecte (20 pts)
    if row["hour"] in heure_suspecte:
        score += 20

    # Piste B — burst user (25 pts)
    if row["user_id"] in burst_users:
        score += 15
    if row["user_id"] in cumul_users:
        score += 10

    # Piste C — geo suspecte (25 pts)
    if row["transaction_id"] in geo_txns:
        score += 25

    # Piste D — nouveau compte montant eleve (30 pts)
    if row["transaction_id"] in new_account_txns:
        score += 30

    return builtins.min(score, 100)

df_score["risk_score"] = df_score.apply(compute_score, axis=1)

top50 = df_score.sort_values("risk_score", ascending=False).head(50)

print("TOP 50 TRANSACTIONS SUSPECTES")
print(top50[["transaction_id", "user_id", "amount_eur", "payment_method",
             "delivery_country", "risk_score"]].to_string(index=False))
print(f"\nScore moyen top 50 : {top50['risk_score'].mean():.1f}")

TOP 50 TRANSACTIONS SUSPECTES
transaction_id  user_id  amount_eur payment_method delivery_country  risk_score
     TXN-01164 USR-0175      597.40 carte_prepayee               UA         100
     TXN-01468 USR-0087     2061.18 carte_prepayee               RO          85
     TXN-00049 USR-0389     1346.64         crypto               BY          85
     TXN-00868 USR-0267     1909.86 carte_prepayee               RO          85
     TXN-01166 USR-0148     1451.91         crypto               UA          85
     TXN-02024 USR-0421     1784.39 carte_prepayee               BY          85
     TXN-01020 USR-0175      593.12 carte_prepayee               XK          80
     TXN-04539 USR-0078      516.09 carte_prepayee               BY          80
     TXN-01874 USR-0194      525.26 carte_prepayee               RO          80
     TXN-04122 USR-0063      571.00 carte_prepayee               UA          80
     TXN-03329 USR-0194      565.68 carte_prepayee               BY          80
     TXN-0

In [ ]:
# Piste E : visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des scores
ax1.hist(df_score["risk_score"], bins=10, color="steelblue", edgecolor="black")
ax1.axvline(x=65, color="red", linestyle="--", label="Seuil top 50")
ax1.set_title("Piste E — Distribution des scores de risque")
ax1.set_xlabel("Score de risque")
ax1.set_ylabel("Nombre de transactions")
ax1.legend()

# Top 10 par score
top10 = top50.head(10)
ax2.barh(top10["transaction_id"], top10["risk_score"], color="tomato")
ax2.axvline(x=75, color="black", linestyle="--", label="Seuil 75")
ax2.set_title("Piste E — Top 10 transactions suspectes")
ax2.set_xlabel("Score de risque")
ax2.legend()

plt.tight_layout()
plt.savefig("piste_e_score.png", dpi=150)
plt.show()
print("Graphique sauvegarde : piste_e_score.png")

Graphique sauvegarde : piste_e_score.png


In [26]:
# Optimisation : mesure cache
import time

# Sans cache
df_no_cache = spark.createDataFrame(pdf)
df_no_cache = df_no_cache.filter(
    (col("amount_eur") > 0) & (col("amount_eur") <= 10000)
)

start = time.time()
df_no_cache.groupBy("payment_method").count().collect()
df_no_cache.groupBy("country_code").count().collect()
df_no_cache.groupBy("delivery_country").count().collect()
time_no_cache = time.time() - start
print(f"Sans cache : {time_no_cache:.2f} secondes")

# Avec cache
df_cached = spark.createDataFrame(pdf)
df_cached = df_cached.filter(
    (col("amount_eur") > 0) & (col("amount_eur") <= 10000)
)
df_cached.cache()
df_cached.count()  # materialiser le cache

start = time.time()
df_cached.groupBy("payment_method").count().collect()
df_cached.groupBy("country_code").count().collect()
df_cached.groupBy("delivery_country").count().collect()
time_cache = time.time() - start
print(f"Avec cache : {time_cache:.2f} secondes")

gain = ((time_no_cache - time_cache) / time_no_cache) * 100
print(f"Gain : {gain:.1f}%")

Sans cache : 1.35 secondes
Avec cache : 0.25 secondes
Gain : 81.2%


In [27]:
# Validation finale vs is_fraud
top50_ids = set(top50["transaction_id"].tolist())

df_validation = df_score[df_score["transaction_id"].isin(top50_ids)]

vrais_positifs = df_validation[df_validation["is_fraud"] == True].shape[0]
faux_positifs = df_validation[df_validation["is_fraud"] == False].shape[0]

precision = vrais_positifs / 50 * 100

print(f"Vrais positifs  : {vrais_positifs} / 50")
print(f"Faux positifs   : {faux_positifs} / 50")
print(f"Precision       : {precision:.1f}%")

# Distribution is_fraud dans le dataset complet
total_fraud = df_score[df_score["is_fraud"] == True].shape[0]
print(f"\nTotal fraudes dans le dataset : {total_fraud} / {len(df_score)}")
print(f"Taux fraude reel : {total_fraud/len(df_score)*100:.1f}%")

Vrais positifs  : 50 / 50
Faux positifs   : 0 / 50
Precision       : 100.0%

Total fraudes dans le dataset : 150 / 4962
Taux fraude reel : 3.0%
